# 06 - Locais críticos e características
Responde:
- As regiões com mais acidentes também têm mais acidentes graves?
- Quais características aparecem nos locais mais críticos?

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # pra achar a pasta src

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

In [ ]:
df = gpd.read_parquet(ARQ_LIMPO)
grade = gpd.read_parquet(PROCESSED / "05_grade.parquet")

## Concentração x gravidade

In [ ]:
grade["vitimas_graves"] = grade["mortos"]
grade["pct_graves"] = grade["mortos"] / grade["acidentes"] * 100

# só células com volume mínimo, senão a porcentagem engana
g = grade[grade["acidentes"] >= 30]
print(len(g), "células com 30+ acidentes")
print("correlação (spearman) acidentes x mortos:", g["acidentes"].corr(g["mortos"], method="spearman").round(3))

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(g["acidentes"], g["mortos"], alpha=0.5)
plt.xlabel("acidentes na célula"); plt.ylabel("mortos"); plt.title("Concentração x mortes")
plt.tight_layout(); plt.savefig(FIGURAS / "acidentes_vs_mortos.png", dpi=150); plt.show()

Ordenar por **taxa** de acidentes com morte mostra locais perigosos que não são necessariamente os mais movimentados.

In [ ]:
g.sort_values("pct_graves", ascending=False).head(10)[["acidentes", "mortos", "pct_graves"]]

## Definindo os hotspots

In [ ]:
corte = grade["acidentes"].quantile(0.95)
hot = grade[grade["acidentes"] >= corte]
print(f"corte (percentil 95): {corte:.0f} acidentes; {len(hot)} células")

dfm = df.to_crs(grade.crs)
dentro = gpd.sjoin(dfm, hot[["geometry"]], how="inner", predicate="within")
dentro = dentro[~dentro.index.duplicated()]
df["HOTSPOT"] = df.index.isin(dentro.index)
df["HOTSPOT"].mean()

## Características dos hotspots vs resto

In [ ]:
def comparar(col):
    t = pd.crosstab(df[col], df["HOTSPOT"], normalize="columns") * 100
    t.columns = ["resto (%)", "hotspot (%)"]
    t["diferenca"] = t["hotspot (%)"] - t["resto (%)"]
    return t.sort_values("diferenca", ascending=False).round(1)

for c in ["TIPOCRUZAMENTO", "CONTROLETRAFEGO", "USOSOLO", "NATUREZA", "SEVERIDADE", "PERIODO"]:
    print(c)
    display(comparar(c).head(8))

In [ ]:
df.groupby("HOTSPOT")[["AUTOMOVEL", "MOTOCICLETA", "BICICLETA", "ONIBUS", "CAMINHAO"]].mean().round(3)

## Conclusões finais
(preencher: resumir as respostas de cada pergunta derivada)